### Imports

In [1]:
from fundus_dataset import AugmentPair, FundusVesselDataset
from torch.utils.data import DataLoader
import torch.nn as nn
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Subset
from monai.networks.nets import UNet
from monai.losses import DiceLoss, SoftclDiceLoss
from safetensors.torch import save_file
from pathlib import Path
from common_utils import get_datasets, get_dataloaders, train_model, make_unet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


### Testing the Dataset

In [2]:
img_dir = "fundus/train/Original/"
mask_dir = "fundus/train/Ground truth"

train_full, val_full = get_datasets(img_dir, mask_dir, transform=AugmentPair(crop_size=(512, 512)))
train_loader, val_loader = get_dataloaders(train_full, val_full, batch_size=8, num_workers=8)


## Sanity checking
images, masks = next(iter(train_loader))
val_images, val_masks = next(iter(val_loader))

print("Train images:", images.shape, images.dtype, images.min().item(), images.max().item())
print("Train masks: ", masks.shape, masks.dtype, torch.unique(masks))
print("Val images:", val_images.shape, val_images.dtype, val_images.min().item(), val_images.max().item())
print("Val masks: ", val_masks.shape, val_masks.dtype, torch.unique(val_masks))

Number of images: 600
Number of masks: 600
First 5 image files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
First 5 mask files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
Number of images: 600
Number of masks: 600
First 5 image files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
First 5 mask files:  ['100_A.png', '101_A.png', '102_A.png', '103_A.png', '104_A.png']
Train size: 480
Val size: 120


Train images: torch.Size([8, 3, 512, 512]) torch.float32 0.0 1.0
Train masks:  torch.Size([8, 1, 512, 512]) torch.float32 tensor([0., 1.])
Val images: torch.Size([1, 3, 2048, 2048]) torch.float32 0.0 1.0
Val masks:  torch.Size([1, 1, 2048, 2048]) torch.float32 tensor([0., 1.])


### Dice + BCE + SoftclDice (MONAI)

In [3]:
class DiceBCEclDiceLogitsLoss(nn.Module):
    """
    Combined Dice + BCE + Soft clDice loss for binary segmentation from logits.

    - DiceLoss handles logits directly via sigmoid=True.
    - BCEWithLogitsLoss is numerically stable on raw logits.
    - SoftclDiceLoss expects 2-channel probability tensors (background, foreground)
      because it internally skips channel 0 with [:, 1:, ...].
    """

    def __init__(self, iter_=3, w_dice=1.0, w_bce=1.0, w_cldice=1.0, smooth=1.0):
        super().__init__()
        self.dice = DiceLoss(sigmoid=True, smooth_nr=smooth, smooth_dr=smooth)
        self.bce = nn.BCEWithLogitsLoss()
        self.cldice = SoftclDiceLoss(iter_=iter_, smooth=smooth)
        self.w_dice = w_dice
        self.w_bce = w_bce
        self.w_cldice = w_cldice

    def forward(self, logits, masks):
        dice_loss = self.dice(logits, masks)
        bce_loss = self.bce(logits, masks)

        pred = torch.sigmoid(logits)
        pred_2ch = torch.cat([1.0 - pred, pred], dim=1)
        true_2ch = torch.cat([1.0 - masks, masks], dim=1)
        cldice_loss = self.cldice(true_2ch, pred_2ch)

        return self.w_dice * dice_loss + self.w_bce * bce_loss + self.w_cldice * cldice_loss

### DANGEROUS (RESET EXPERIMENT)

In [4]:
model = make_unet(channels=(32, 64, 128, 256, 512))
balanced_clDice = DiceBCEclDiceLogitsLoss(iter_=10, w_dice=1/3, w_bce=1/3, w_cldice=1/3, smooth=1.0).to(device)
dice_dominates = DiceBCEclDiceLogitsLoss(iter_=10, w_dice=0.6, w_bce=0.2, w_cldice=0.2, smooth=1.0).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

### START OR CONTINUE EXPERIMENT

In [5]:
balanced_clDice_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=balanced_clDice,
    optimizer=optimizer,
    epochs=100,
    save_path="/workspace/models_clDice/balanced_clDice.pt",
)

dice_dominates_history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=dice_dominates,
    optimizer=optimizer,
    epochs=100,
    save_path="/workspace/models_clDice/dice_dominates.pt",
)

Epoch 001/100 | Train loss: 0.6371 | Val loss: 0.6136 | Val Dice: 0.6175 | Val clDice: 0.5174 | Best: 0.6175 @ 1
Epoch 002/100 | Train loss: 0.5614 | Val loss: 0.5748 | Val Dice: 0.6379 | Val clDice: 0.6695 | Best: 0.6379 @ 2
Epoch 003/100 | Train loss: 0.5218 | Val loss: 0.5492 | Val Dice: 0.6985 | Val clDice: 0.7354 | Best: 0.6985 @ 3
Epoch 004/100 | Train loss: 0.4958 | Val loss: 0.5176 | Val Dice: 0.7377 | Val clDice: 0.7914 | Best: 0.7377 @ 4
Epoch 005/100 | Train loss: 0.4796 | Val loss: 0.5087 | Val Dice: 0.7568 | Val clDice: 0.7883 | Best: 0.7568 @ 5
Epoch 006/100 | Train loss: 0.4533 | Val loss: 0.4727 | Val Dice: 0.7678 | Val clDice: 0.8174 | Best: 0.7678 @ 6
Epoch 007/100 | Train loss: 0.4345 | Val loss: 0.4743 | Val Dice: 0.7015 | Val clDice: 0.7747 | Best: 0.7678 @ 6
Epoch 008/100 | Train loss: 0.4242 | Val loss: 0.4531 | Val Dice: 0.7484 | Val clDice: 0.8121 | Best: 0.7678 @ 6


KeyboardInterrupt: 

### Save your Settings

In [ ]:
import pandas as pd

balanced_clDice_history_df = pd.DataFrame(balanced_clDice_history)
balanced_clDice_history_df.to_csv("/workspace/models_clDice/history_balanced_clDice.csv", index=False)

dice_dominates_history_df = pd.DataFrame(dice_dominates_history)
dice_dominates_history_df.to_csv("/workspace/models_clDice/history_dice_dominates.csv", index=False)

